# Creación del Dataset para ForeCasting

In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

### Cargamos el dataset del paso anterior

In [3]:
df = pd.read_csv('df_forecast_raw.csv')
display(df.head())

,IES,DEPARTAMENTO,MUNICIPIO,CARACTER,ORIGEN,periodo,DESERTORES,MATRICULADOS,tasa
0,1101,bogota d c,bogota dc,Universidad,Oficial,1998-1,0.0,2219.0,0.00
1,1101,bogota d c,bogota dc,Universidad,Oficial,1998-2,0.0,4118.0,0.00
2,1101,bogota d c,bogota dc,Universidad,Oficial,1999-1,151.0,6079.0,2.48
3,1101,bogota d c,bogota dc,Universidad,Oficial,1999-2,396.0,7934.0,4.99
4,1101,bogota d c,bogota dc,Universidad,Oficial,2000-1,469.0,9926.0,4.72


Organizamos los datos

In [4]:
df = df.sort_values(['IES','periodo']).reset_index(drop= True)
df['periodo_num'] = (df['periodo'].str.split('-').str[0].astype(int) * 2
                     + df['periodo'].str.split('-').str[1].astype(int) - 1)
display(df.head())

,IES,DEPARTAMENTO,MUNICIPIO,CARACTER,ORIGEN,periodo,DESERTORES,MATRICULADOS,tasa,periodo_num
0,1101,bogota d c,bogota dc,Universidad,Oficial,1998-1,0.0,2219.0,0.00,3996
1,1101,bogota d c,bogota dc,Universidad,Oficial,1998-2,0.0,4118.0,0.00,3997
2,1101,bogota d c,bogota dc,Universidad,Oficial,1999-1,151.0,6079.0,2.48,3998
3,1101,bogota d c,bogota dc,Universidad,Oficial,1999-2,396.0,7934.0,4.99,3999
4,1101,bogota d c,bogota dc,Universidad,Oficial,2000-1,469.0,9926.0,4.72,4000


Ventana deslizante

In [44]:
filas = []
for ies, grupo in df.groupby('IES'):
    tasas    = grupo['tasa'].values
    periodos = grupo['periodo'].values
    for i in range(4, len(tasas)):        # ya no hay -1
        fila = {
            'IES':            ies,
            'lag4':           tasas[i-4],
            'lag3':           tasas[i-3],
            'lag2':           tasas[i-2],
            'lag1':           tasas[i-1],
            'target':         tasas[i],   # un solo target
            'periodo_target': periodos[i],
        }
        filas.append(fila)

dataset = pd.DataFrame(filas)
print(dataset.shape)

display(dataset.head())

(12560, 7)


,IES,lag4,lag3,lag2,lag1,target,periodo_target
0,1101,0.00,0.00,2.48,4.99,4.72,2000-1
1,1101,0.00,2.48,4.99,4.72,4.79,2000-2
2,1101,2.48,4.99,4.72,4.79,5.52,2001-1
3,1101,4.99,4.72,4.79,5.52,4.55,2001-2
4,1101,4.72,4.79,5.52,4.55,4.98,2002-1


Agregamos los datos categoricos

In [45]:

metadata = df.groupby('IES')[['CARACTER','ORIGEN','DEPARTAMENTO','MUNICIPIO']].first().reset_index()

dataset = dataset.merge(metadata, on='IES', how='left')
display(dataset.head())


,IES,lag4,lag3,lag2,lag1,target,periodo_target,CARACTER,ORIGEN,DEPARTAMENTO,MUNICIPIO
0,1101,0.00,0.00,2.48,4.99,4.72,2000-1,Universidad,Oficial,bogota d c,bogota dc
1,1101,0.00,2.48,4.99,4.72,4.79,2000-2,Universidad,Oficial,bogota d c,bogota dc
2,1101,2.48,4.99,4.72,4.79,5.52,2001-1,Universidad,Oficial,bogota d c,bogota dc
3,1101,4.99,4.72,4.79,5.52,4.55,2001-2,Universidad,Oficial,bogota d c,bogota dc
4,1101,4.72,4.79,5.52,4.55,4.98,2002-1,Universidad,Oficial,bogota d c,bogota dc


### Verificamos Outliers

In [46]:
display(dataset.describe().round(2))

,IES,lag4,lag3,lag2,lag1,target
count,12560.00,12560.00,12560.00,12560.00,12560.00,12560.00
mean,2901.17,28.83,29.09,29.34,28.85,28.79
std,1847.36,1247.10,1247.09,1247.09,1246.48,1246.47
min,1101.00,0.00,0.00,0.00,0.00,0.00
25%,1726.00,6.39,6.63,6.85,6.93,6.97
50%,2715.00,9.57,9.68,9.81,9.86,9.89
75%,3706.00,14.22,14.29,14.44,14.47,14.49
max,9914.00,138700.00,138700.00,138700.00,138700.00,138700.00


In [48]:
print((dataset[['lag4','lag3','lag2','lag1','target']] > 100).sum())

lag4      148
lag3      149
lag2      149
lag1      138
target    130
dtype: int64


In [49]:
#Revisamos NaN
print(dataset.isna().sum())

IES               0
lag4              0
lag3              0
lag2              0
lag1              0
target            0
periodo_target    0
CARACTER          0
ORIGEN            0
DEPARTAMENTO      0
MUNICIPIO         0
dtype: int64


#### Procedemos a corregir los casos donde los valores superan el 100% de la tasa que vimos anteiormente 

In [50]:
dataset[['lag4','lag3','lag2','lag1','target']] = np.clip((dataset[['lag4','lag3','lag2','lag1','target']]),0,100)

In [52]:
print((dataset[['lag4','lag3','lag2','lag1','target']] > 100).sum())

lag4      0
lag3      0
lag2      0
lag1      0
target    0
dtype: int64


In [53]:
display(dataset.describe())

,IES,lag4,lag3,lag2,lag1,target
count,12560.000000,12560.000000,12560.000000,12560.000000,12560.000000,12560.000000
mean,2901.173487,13.047875,13.297471,13.544527,13.496792,13.481839
std,1847.356836,14.637983,14.608616,14.527931,14.194232,13.985076
min,1101.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1726.000000,6.390000,6.630000,6.850000,6.930000,6.970000
50%,2715.000000,9.570000,9.680000,9.810000,9.860000,9.885000
75%,3706.000000,14.220000,14.292500,14.440000,14.470000,14.492500
max,9914.000000,100.000000,100.000000,100.000000,100.000000,100.000000


### Split de train, validation y test

In [55]:
train = dataset[dataset['periodo_target'] <= '2023-1']
val   = dataset[dataset['periodo_target'] == '2023-2']
test  = dataset[dataset['periodo_target'].isin(['2024-1', '2024-2'])]

print(train.shape, val.shape, test.shape)


(11766, 11) (265, 11) (529, 11)


Convertimos a categorical las columnas categoricas

In [56]:
cols_cat = ['CARACTER', 'ORIGEN', 'DEPARTAMENTO', 'MUNICIPIO']

for col in cols_cat:
    train[col] = train[col].astype('category')
    val[col]   = val[col].astype('category')
    test[col]  = test[col].astype('category')
display(train.info())

<class 'pandas.core.frame.DataFrame'>
Index: 11766 entries, 0 to 12556
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   IES             11766 non-null  int64   
 1   lag4            11766 non-null  float64 
 2   lag3            11766 non-null  float64 
 3   lag2            11766 non-null  float64 
 4   lag1            11766 non-null  float64 
 5   target          11766 non-null  float64 
 6   periodo_target  11766 non-null  object  
 7   CARACTER        11766 non-null  category
 8   ORIGEN          11766 non-null  category
 9   DEPARTAMENTO    11766 non-null  category
 10  MUNICIPIO       11766 non-null  category
dtypes: category(4), float64(5), int64(1), object(1)
memory usage: 785.5+ KB


C:\Users\Jonathan Piedrahita\AppData\Local\Temp\ipykernel_32644\434128757.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train[col] = train[col].astype('category')
C:\Users\Jonathan Piedrahita\AppData\Local\Temp\ipykernel_32644\434128757.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val[col]   = val[col].astype('category')
C:\Users\Jonathan Piedrahita\AppData\Local\Temp\ipykernel_32644\434128757.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try 

None

Convertimos el periodo en semestre

In [57]:
train['semestre'] = train['periodo_target'].str.split('-').str[1].astype(int)
val['semestre'] = val['periodo_target'].str.split('-').str[1].astype(int)
test['semestre'] = test['periodo_target'].str.split('-').str[1].astype(int)
display(val.info())

<class 'pandas.core.frame.DataFrame'>
Index: 265 entries, 47 to 12557
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   IES             265 non-null    int64   
 1   lag4            265 non-null    float64 
 2   lag3            265 non-null    float64 
 3   lag2            265 non-null    float64 
 4   lag1            265 non-null    float64 
 5   target          265 non-null    float64 
 6   periodo_target  265 non-null    object  
 7   CARACTER        265 non-null    category
 8   ORIGEN          265 non-null    category
 9   DEPARTAMENTO    265 non-null    category
 10  MUNICIPIO       265 non-null    category
 11  semestre        265 non-null    int64   
dtypes: category(4), float64(5), int64(2), object(1)
memory usage: 23.8+ KB


C:\Users\Jonathan Piedrahita\AppData\Local\Temp\ipykernel_32644\481311707.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['semestre'] = train['periodo_target'].str.split('-').str[1].astype(int)
C:\Users\Jonathan Piedrahita\AppData\Local\Temp\ipykernel_32644\481311707.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val['semestre'] = val['periodo_target'].str.split('-').str[1].astype(int)
C:\Users\Jonathan Piedrahita\AppData\Local\Temp\ipykernel_32644\481311707.py:3: SettingWithCopyWarning: 
A v

None

Entrenamiento X y Y

In [59]:
features = ['lag4', 'lag3', 'lag2', 'lag1','semestre'] + cols_cat

X_train = train[features]
y_train = train['target']

X_val = val[features]
y_val = val['target']

X_test = test[features]
y_test = test['target']


In [42]:
display(X_train.head(20))

,lag4,lag3,lag2,lag1,semestre,CARACTER,ORIGEN,DEPARTAMENTO,MUNICIPIO
0,0.00,0.00,2.48,4.99,1,Universidad,Oficial,bogota d c,bogota dc
1,0.00,2.48,4.99,4.72,2,Universidad,Oficial,bogota d c,bogota dc
2,2.48,4.99,4.72,4.79,1,Universidad,Oficial,bogota d c,bogota dc
3,4.99,4.72,4.79,5.52,2,Universidad,Oficial,bogota d c,bogota dc
4,4.72,4.79,5.52,4.55,1,Universidad,Oficial,bogota d c,bogota dc
5,4.79,5.52,4.55,4.98,2,Universidad,Oficial,bogota d c,bogota dc
6,5.52,4.55,4.98,4.82,1,Universidad,Oficial,bogota d c,bogota dc
7,4.55,4.98,4.82,5.04,2,Universidad,Oficial,bogota d c,bogota dc
8,4.98,4.82,5.04,7.30,1,Universidad,Oficial,bogota d c,bogota dc
9,4.82,5.04,7.30,1.36,2,Universidad,Oficial,bogota d c,bogota dc


## Guardamos los datasets creados

In [60]:
parent = Path('entrenamiento_csv')
parent.mkdir(exist_ok=True)
X_train.to_csv(parent / 'X_train.csv',index=False)
y_train.to_csv(parent / 'y_train.csv',index=False)
X_val.to_csv(parent / 'X_val.csv',index=False)
y_val.to_csv(parent / 'y_val.csv',index=False)
X_test.to_csv(parent / 'X_test.csv',index=False)
y_test.to_csv(parent / 'y_test.csv',index=False)